In [1]:
!pip install -q langchain langchain-google-genai python-dotenv


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

# 1. Load local .env if present
load_dotenv(find_dotenv())

# 2. Load from Google Colab Secrets if running in Colab
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = (
            userdata.get("GOOGLE_API_KEY") or userdata.get("GEMINI_API_KEY")
        )
        print("✅ Loaded GOOGLE_API_KEY from Google Colab Secrets.")
    except Exception:
        pass

# 3. Interactive fallback if still missing
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google Gemini API key: ")

import warnings
from langchain_google_genai import ChatGoogleGenerativeAI

warnings.filterwarnings(
    "ignore",
    message=r"Model 'gemini-3\.5-flash-lite' uses fixed sampling defaults.*",
)

MODEL_NAME = "gemini-3.5-flash-lite"
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0.3, max_retries=3, timeout=30)
print(f"🚀 Gemini LLM ready: {MODEL_NAME} (temperature=0.3)")

🚀 Gemini LLM ready: gemini-3.5-flash-lite (temperature=0.3)


In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer.
        Write prices as plain numbers without commas (e.g. Rs 1800)."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! Yes, all our cakes have eggless versions! Just remember that orders need 24 hours' notice. Let me know what you'd like to order!


Your name is Ravi! How can I help you with your order today?


In [4]:
customer_questions = [
    "Do you have eggless cake options?",
    "Can I get a cake today evening?",
    "Do you deliver to Kakinada?",
    "What is the price of 6 cupcakes?",
]

for q in customer_questions:
    print(f"Customer: {q}")
    print(f"Assistant: {ask(q)}\n")

Customer: Do you have eggless cake options?


Assistant: Hi Ravi! Yes, all our cakes have eggless versions. Plum cake is Rs 500, chocolate cake is Rs 650, and cupcakes are Rs 80 each. Orders need 24 hours' notice. Would you like to place an order?

Customer: Can I get a cake today evening?


Assistant: Hi Ravi! Orders need 24 hours' notice, so I can't take a booking for this evening. I'll check with the owner about any exceptions, or we can plan for tomorrow!

Customer: Do you deliver to Kakinada?


Assistant: Hi Ravi! I'll check with the owner about delivering to Kakinada.

Customer: What is the price of 6 cupcakes?


Assistant: Hi Ravi! 6 cupcakes will cost Rs 480. Orders need 24 hours' notice. Let me know if you'd like to place an order!



In [5]:
import os

TESTS_CSV = """question,must_include
How much is a plum cake?,500
Do you make eggless cakes?,eggless
Can I get a cake today evening?,24 hours
Do you deliver to Kakinada?,check with the owner
What is the price of 6 cupcakes?,480
How much does a chocolate cake cost?,650
Is there an eggless version of the chocolate cake?,eggless
What is the total for 1 plum cake and 2 cupcakes?,660
Do you sell sugar-free cookies?,check with the owner
What is the total cost of 2 chocolate cakes and 1 plum cake?,1800
"""

if not os.path.exists("tests.csv"):
    with open("tests.csv", "w", encoding="utf-8") as f:
        f.write(TESTS_CSV)
    print("✅ Created tests.csv")
else:
    print("ℹ️ Using existing tests.csv")

ℹ️ Using existing tests.csv


In [6]:
import csv
import time

def run_tests(path: str = "tests.csv", test_chain=None, delay: float = 0.5):
    active_chain = test_chain or chain
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    passed = 0
    answers = []
    for row in rows:
        for attempt in range(4):
            try:
                answer = active_chain.invoke({"history": [], "question": row["question"]})
                break
            except (KeyError, ValueError, FileNotFoundError):
                raise
            except Exception as err:
                if attempt == 3:
                    raise
                wait_s = 2 ** (attempt + 1)
                print(f"   ⏳ Transient error ({type(err).__name__}), retrying in {wait_s}s...")
                time.sleep(wait_s)
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        answers.append(answer)
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
        if delay:
            time.sleep(delay)
    rate = passed / len(rows)
    print(f"\n{passed}/{len(rows)} passed ({rate:.0%})")
    return rate, answers

bakery_rate, _ = run_tests("tests.csv")

PASS | How much is a plum cake? → Hi! A delicious plum cake is just Rs 500. We need 24 hours' notice, an


PASS | Do you make eggless cakes? → Yes, we do! All our cakes have eggless versions. Would you like to pla


PASS | Can I get a cake today evening? → Hello! Orders need 24 hours' notice, so I can't take a same-day order 


PASS | Do you deliver to Kakinada? → Hello! I'll check with the owner about delivery to Kakinada. Let me kn


PASS | What is the price of 6 cupcakes? → Hello! Six cupcakes would be Rs 480. Please note that orders need 24 h


PASS | How much does a chocolate cake cost? → Hello! A delicious chocolate cake costs Rs 650, and we have an eggless


... (2 outputs left out)

PASS | Do you sell sugar-free cookies? → I'll check with the owner! Let me know if you'd like to try our delici


   ⏳ Transient error (GoogleRateLimitError), retrying in 2s...


PASS | What is the total cost of 2 chocolate cakes and 1 plum cake? → Hello! Two chocolate cakes and one plum cake come to a total of Rs 180



10/10 passed (100%)


In [7]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come on Day 4
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].text)

Yes, absolutely! All our delicious cakes have eggless versions. Would you like to place an order? Please note that orders need 24 hours notice.


### 1. Your business — BrightPage Stationery `SYSTEM` & Chain

In [8]:
STATIONERY_CATALOGUE = {
    "spiral notebook (200 pgs)": 85,
    "a4 copier paper ream (500 sheets)": 320,
    "gel pen pack (10 pens, blue/black)": 150,
    "pastel highlighter set (6 colors)": 180,
    "heavy-duty stapler + pins": 240,
    "sticky notes pack (400 sheets)": 95,
    "geometry box set": 160,
    "whiteboard marker set (4 colors)": 120,
    "permanent marker (black)": 35,
    "clear file folder (pack of 5)": 110,
}

CATALOGUE_TEXT = "\n".join(
    f"  - {item}: Rs {price}" for item, price in STATIONERY_CATALOGUE.items()
)

BRIGHTPAGE_SYSTEM_V1 = f"""ROLE:   You are the friendly assistant for BrightPage Stationery & Office Supplies.
FACTS:  Catalogue (INR):
{CATALOGUE_TEXT}
        Hours: Mon-Sat 9:00 AM - 8:30 PM, Sunday 10:00 AM - 4:00 PM.
        Delivery: Free local delivery on orders above Rs 500; Rs 40 fee below Rs 500.
        Bulk discount: 10% off on 10+ identical items or orders above Rs 1,000.
        Returns: Damaged items replaced within 3 days with receipt.
RULES:  Never promise delivery outside local city limits. If an item isn't in FACTS,
        say "I'll check with the store manager about availability and get back to you shortly."
        Work out totals carefully.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

bp_prompt_v1 = ChatPromptTemplate.from_messages([
    ("system", BRIGHTPAGE_SYSTEM_V1),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
bp_chain_v1 = bp_prompt_v1 | llm | StrOutputParser()
print("✅ BrightPage baseline chain (V1) ready.")

✅ BrightPage baseline chain (V1) ready.


In [9]:
BRIGHTPAGE_TESTS_CSV = """question,must_include
How much is a spiral notebook?,85
What is the price of 2 A4 copier paper reams?,640
Are you open on Sunday?,10:00
Is delivery free for an order of Rs 600?,free
What is the delivery fee for a Rs 300 order?,40
Do you sell scientific calculators?,store manager
What is your return policy for damaged items?,3 days
What discount do I get if I buy 10 gel pen packs?,10%
How much is 1 geometry box set and 1 sticky notes pack?,255
Do you deliver outside the city?,city
"""

if not os.path.exists("brightpage_tests.csv"):
    with open("brightpage_tests.csv", "w", encoding="utf-8") as f:
        f.write(BRIGHTPAGE_TESTS_CSV)
    print("✅ Created brightpage_tests.csv")
else:
    print("ℹ️ Using existing brightpage_tests.csv")

print("\n=== Baseline Prompt (V1) ===")
v1_rate, v1_answers = run_tests("brightpage_tests.csv", test_chain=bp_chain_v1)

ℹ️ Using existing brightpage_tests.csv

=== Baseline Prompt (V1) ===


PASS | How much is a spiral notebook? → Hello! A 200-page spiral notebook is Rs 85 at BrightPage Stationery & 


PASS | What is the price of 2 A4 copier paper reams? → Hello! A single A4 copier paper ream (500 sheets) is Rs 320. 

For 2 r


PASS | Are you open on Sunday? → Yes, we are open on Sundays from 10:00 AM to 4:00 PM! (Our Monday to S


PASS | Is delivery free for an order of Rs 600? → Yes, absolutely! 🚚 Delivery is completely free for local orders above 


PASS | What is the delivery fee for a Rs 300 order? → Hi there! For an order of Rs 300, the local delivery fee is just Rs 40


... (2 outputs left out)

PASS | What discount do I get if I buy 10 gel pen packs? → Hello! You get a fantastic 10% bulk discount when you buy 10 identical


PASS | How much is 1 geometry box set and 1 sticky notes pack? → Hello! A geometry box set is Rs 160 and a sticky notes pack is Rs 95. 


PASS | Do you deliver outside the city? → Hello! I'm sorry, but we only offer local delivery and cannot deliver 



10/10 passed (100%)


In [10]:
# Change one thing: write amounts without commas, mention 'local city limits' when relevant, and stay focused
BRIGHTPAGE_SYSTEM_V2 = BRIGHTPAGE_SYSTEM_V1 + """
        Write prices without commas (e.g. Rs 1800), mention 'local city limits' when asked
        about out-of-city delivery, and answer only what the customer asked."""

bp_prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", BRIGHTPAGE_SYSTEM_V2),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
bp_chain_v2 = bp_prompt_v2 | llm | StrOutputParser()

print("=== Improved Prompt (V2) ===")
v2_rate, v2_answers = run_tests("brightpage_tests.csv", test_chain=bp_chain_v2)
print(f"\nPass rate change: {v1_rate:.0%} -> {v2_rate:.0%}")

=== Improved Prompt (V2) ===


PASS | How much is a spiral notebook? → Hello! A 200-page spiral notebook is just Rs 85. Let me know if you ne


PASS | What is the price of 2 A4 copier paper reams? → Hello! Two A4 copier paper reams (500 sheets each) cost Rs 640. (Since


PASS | Are you open on Sunday? → Hello! Yes, we are open on Sundays from 10:00 AM to 4:00 PM. We look f


PASS | Is delivery free for an order of Rs 600? → Yes, delivery is completely free for your order of Rs 600 since it is 


PASS | What is the delivery fee for a Rs 300 order? → Hello! The delivery fee for a Rs 300 order is Rs 40. We offer free loc


... (2 outputs left out)

PASS | What discount do I get if I buy 10 gel pen packs? → Hello! You get a wonderful 10% bulk discount when you buy 10 identical


PASS | How much is 1 geometry box set and 1 sticky notes pack? → Hello! One geometry box set is Rs 160 and one sticky notes pack is Rs 


PASS | Do you deliver outside the city? → Hello! We only offer delivery within local city limits. We'd love to h



10/10 passed (100%)

Pass rate change: 100% -> 100%


In [11]:
llm_hot = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=1.0, max_retries=3, timeout=30)
bp_chain_hot = bp_prompt_v2 | llm_hot | StrOutputParser()

print("=== Temperature 1.0 — Run 1 ===")
hot_rate_1, hot_answers_1 = run_tests("brightpage_tests.csv", test_chain=bp_chain_hot)

print("\n=== Temperature 1.0 — Run 2 ===")
hot_rate_2, hot_answers_2 = run_tests("brightpage_tests.csv", test_chain=bp_chain_hot)

changed = sum(a != b for a, b in zip(hot_answers_1, hot_answers_2))
print(f"\nChanged answers between Run 1 and Run 2: {changed}/{len(hot_answers_1)}")
print(f"Pass rates: Run 1 = {hot_rate_1:.0%}, Run 2 = {hot_rate_2:.0%}")

=== Temperature 1.0 — Run 1 ===


PASS | How much is a spiral notebook? → Hello! A 200-page spiral notebook is Rs 85. Let me know if you need an


PASS | What is the price of 2 A4 copier paper reams? → Hello! Two A4 copier paper reams (500 sheets each) cost Rs 640. (Since


PASS | Are you open on Sunday? → Yes, we are open on Sundays from 10:00 AM to 4:00 PM! Drop by whenever


PASS | Is delivery free for an order of Rs 600? → Yes, delivery is completely free for your order of Rs 600 since it is 


PASS | What is the delivery fee for a Rs 300 order? → Hello! Since your order is below Rs 500, the delivery fee is Rs 40 (wi


... (13 outputs left out)

PASS | What discount do I get if I buy 10 gel pen packs? → Hello! You get a fantastic 10% bulk discount when you buy 10 identical


PASS | How much is 1 geometry box set and 1 sticky notes pack? → Hello! One geometry box set is Rs 160 and one sticky notes pack is Rs 


PASS | Do you deliver outside the city? → Hello! We only deliver within local city limits.



10/10 passed (100%)

Changed answers between Run 1 and Run 2: 7/10
Pass rates: Run 1 = 100%, Run 2 = 100%


In [12]:
business_jobs = [
    (
        "Calculate cart subtotal, apply 10% bulk discount, and add Rs 40 delivery fee",
        "plain code",
        "Deterministic math with fixed business rules needs zero LLM reasoning.",
    ),
    (
        "Classify an incoming WhatsApp message as order, inquiry, or complaint",
        "one LLM call",
        "Single-step natural language classification without external tool calls.",
    ),
    (
        "Check live inventory, compute bulk pricing, and record a confirmed order in the sheet",
        "agent",
        "Multi-step task where the model decides which tools to call based on user input.",
    ),
]

for job, label, reason in business_jobs:
    print(f"[{label.upper()}] {job}\n   Why: {reason}\n")

[PLAIN CODE] Calculate cart subtotal, apply 10% bulk discount, and add Rs 40 delivery fee
   Why: Deterministic math with fixed business rules needs zero LLM reasoning.

[ONE LLM CALL] Classify an incoming WhatsApp message as order, inquiry, or complaint
   Why: Single-step natural language classification without external tool calls.

[AGENT] Check live inventory, compute bulk pricing, and record a confirmed order in the sheet
   Why: Multi-step task where the model decides which tools to call based on user input.

